# 6 · Phasor Neural Networks

*Phasor networks, from the ground up — notebook 6 of 7.*

The same phase algebra trains by gradient descent. A `PhasorDense` layer applies a
complex linear map to phasors and renormalizes to the unit circle; stacking them
gives a phasor MLP that we train with `train()` and standard `Optimisers`. We
classify a synthetic **bullseye** (two concentric rings) — small, self-contained,
no external dataset.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots, Lux
using NNlib: tanh_fast
using Distributions: Normal
using OneHotArrays: onehotbatch, onecold
using Random: Xoshiro
using Statistics: mean

## Synthetic data

Two classes: an inner blob and an outer ring, in 2-D. Not linearly separable.

In [ ]:
function bullseye_data(n_s, rng)
    d = Normal(0.0, 0.08)
    y = rand(rng, (0, 1), n_s)
    r = rand(rng, d, n_s) .+ (0.4 .* y)
    phi = (rand(rng, Float64, n_s) .- 1) .* (2 * pi)
    data = Float32.(cat(r .* cos.(phi), r .* sin.(phi), dims=2)')
    labels = onehotbatch(y, 0:1)
    return data, labels
end

rng = Xoshiro(0)
xv, yv = bullseye_data(2000, rng)
scatter(xv[1, :], xv[2, :], group=vec(onecold(yv)), markersize=2, aspect_ratio=:equal,
        title="bullseye data", xlabel="x", ylabel="y")

## A phasor MLP

Real inputs are squashed into phases with `tanh`, then passed through two `PhasorDense` layers. The output is two phases (one per class).

In [ ]:
model = Chain(x -> Phase.(tanh_fast.(x)),
              PhasorDense(2 => 64, normalize_to_unit_circle, use_bias=true),
              PhasorDense(64 => 2, normalize_to_unit_circle, use_bias=true))
ps, st = Lux.setup(rng, model)

loss(x, y, m, p, s) = mean(evaluate_loss(m(x, p, s)[1], y, :quadrature));

## Train

CPU backend, a few epochs over freshly-sampled batches.

In [ ]:
args = Args(batchsize = 64, epochs = 5, backend = :cpu)
train_loader = [bullseye_data(args.batchsize, rng) for _ in 1:100]
test_loader  = [bullseye_data(args.batchsize, rng) for _ in 1:20]

losses, ps, st = train(model, ps, st, train_loader, loss, args)
_, acc = loss_and_accuracy(test_loader, model, ps, st, args, encoding=:quadrature)
println("final training loss: ", round(losses[end], digits=4))
println("test accuracy: ", acc)

In [ ]:
plot(losses, xlabel="step", ylabel="loss", label="", title="training loss")

## Takeaway

A phasor network is an ordinary differentiable model whose activations are phases
and whose linear maps act on the unit circle. The trained layers are the same
`PhasorDense` operations whose oscillator equivalence we established earlier — so
this network can be run as a bank of spiking oscillators with no retraining. The
finale makes the *time* axis itself carry information.